## 02a — RAG Strategy 1: Context-Only Retrieval

Builds a FAISS index (`IndexFlatL2`, 1,000 vectors, dim 768) over Legal-BERT embeddings (`nlpaueb/legal-bert-base-uncased`) of context chunks from 1,000 CaseHOLD training examples (chunk size 1,000 chars, overlap 100). At inference time, the top-3 retrieved context chunks are passed to `mistralai/Mistral-7B-Instruct-v0.2` to predict the legal holding.

**Note on inference metrics:** The inference loop was interrupted by the Colab session. Evaluation metrics (ROUGE-L and Legal-BERT cosine similarity) in the final cell are computed on a partial sample of approximately 21–29 of the 3,900 validation examples — not the full validation set.

In [1]:
import json
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import random
import subprocess


In [5]:
from rouge_score import rouge_scorer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
from huggingface_hub import notebook_login # hf_REDACTED
notebook_login()

In [7]:
from datasets import load_dataset

dataset = load_dataset("coastalcph/lex_glue", "case_hold")
train_data = dataset["train"]
test_dataset = dataset["validation"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

case_hold/train-00000-of-00001.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

case_hold/test-00000-of-00001.parquet:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

case_hold/validation-00000-of-00001.parq(…):   0%|          | 0.00/3.51M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3600 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3900 [00:00<?, ? examples/s]

In [8]:
# -------------------------
# 2. Prepare RAG retrieval (Option 1: context only)
# -------------------------
from langchain.text_splitter import RecursiveCharacterTextSplitter

train_rows = dataset["train"].select(range(1000))  # first 1000 rows as dicts
docs = [row["context"] for row in train_rows]

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = []
for i, doc in enumerate(docs):
    for chunk in splitter.split_text(doc):
        chunks.append({"id": i, "text": chunk})

chunks_texts = [chunk["text"] for chunk in chunks]


In [17]:
from transformers import AutoTokenizer, AutoModel
import torch
import faiss
import numpy as np

# -------------------------
# 3. Create embeddings and FAISS index using Legal-BERT
# -------------------------
tokenizer_legal = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_legal = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_legal.eval()  # evaluation mode
device = "cuda" if torch.cuda.is_available() else "cpu"
model_legal.to(device)

def get_legal_bert_embedding(text):
    inputs = tokenizer_legal(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model_legal(**inputs)
    # Mean pooling over token embeddings
    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().cpu().numpy()

# Generate embeddings for all chunks
embeddings = []
for chunk_text in chunks_texts:
    emb = get_legal_bert_embedding(chunk_text)
    embeddings.append(emb)
embeddings = np.array(embeddings)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"✅ FAISS index created with {len(embeddings)} embeddings of dimension {dimension}")




# -------------------------
# 3. Create embeddings and FAISS index
# -------------------------
#-----------OLD----
#embedder = SentenceTransformer("all-mpnet-base-v2")#("all-MiniLM-L6-v2")
#embeddings = embedder.encode(chunks_texts, convert_to_numpy=True, show_progress_bar=True)

#dimension = embeddings.shape[1]
#index = faiss.IndexFlatL2(dimension)
#index.add(embeddings)
#--------------


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

✅ FAISS index created with 1000 embeddings of dimension 768


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------
# 1. Load Mistral-7B-Instruct-v0.2
# -------------------------
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # saves memory
    device_map="auto"           # auto-distributes across GPU if available
)

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [18]:

# -------------------------
# 2. Define RAG inference functions
# -------------------------
def retrieve(query, top_k=3):
    query_vec = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, top_k)
    return [chunks[i]["text"] for i in indices[0]]

def ask_mistral(context, query, max_new_tokens=256):
    prompt = f"""
You are a legal assistant.
Use ONLY the retrieved context below to answer the question.
If the holding is present in the context, copy it verbatim.
Do NOT paraphrase, invent, or summarize.

Retrieved Context:
{context}

Question: {query}
Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.1,  # low temp = deterministic
        do_sample=False
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.replace(prompt, "").strip()

def run_rag_inference(query):
    retrieved = retrieve(query, top_k=3)
    context = "\n".join(retrieved)
    answer = ask_mistral(context, query)
    return answer, retrieved

In [19]:



# -------------------------
# 3. Run inference on test set
# -------------------------
results = []

for i, example in enumerate(test_dataset):
    gold = example["endings"][example["label"]]
    query = "What is the holding?"

    answer, docs_used = run_rag_inference(query=query)

    results.append({
        "id": str(i),
        "context": example["context"],
        "gold_holding": gold,
        "predicted_answer": answer
    })

    if i % 10 == 0:
        print(f"Processed {i} examples...")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 0 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 10 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 20 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


KeyboardInterrupt: 

**Note on results:** The inference loop was interrupted before completion. Metrics below are computed on a partial sample of approximately 21–29 of 3,900 validation examples. Results should be read as preliminary, not full-dataset evaluation.

In [20]:
import evaluate
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# -------------------------
# 7. Compute ROUGE-L + Cosine Similarity using Legal-BERT
# -------------------------

# Load Legal-BERT
tokenizer_legal = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_legal = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_legal.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_legal.to(device)

def get_legal_bert_embedding(text):
    inputs = tokenizer_legal(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model_legal(**inputs)
    # Mean pooling
    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().cpu().numpy()

# Initialize ROUGE
rouge = evaluate.load("rouge")

rouge_scores, cosine_scores = [], []

for row in results:
    pred = row["predicted_answer"]
    gold = row["gold_holding"]

    # ROUGE-L
    rouge_result = rouge.compute(predictions=[pred], references=[gold], rouge_types=["rougeL"])
    rouge_l = rouge_result["rougeL"]
    rouge_scores.append(rouge_l)

    # Cosine similarity using Legal-BERT
    emb_pred = get_legal_bert_embedding(pred)
    emb_gold = get_legal_bert_embedding(gold)
    cos_sim = cosine_similarity([emb_pred], [emb_gold])[0][0]
    cosine_scores.append(cos_sim)

# Save metrics CSV
df_metrics = pd.DataFrame({
    "id": [row["id"] for row in results],
    "gold": [row["gold_holding"] for row in results],
    "predicted": [row["predicted_answer"] for row in results],
    "rougeL": rouge_scores,
    "cosine": cosine_scores
})
# df_metrics.to_csv("pipeline1_rag_eval_legalbert.csv", index=False)
print(f"✅ Saved evaluation metrics CSV to pipeline1_rag_eval_legalbert.csv")
print("Avg ROUGE-L:", sum(rouge_scores)/len(rouge_scores))
print("Avg Cosine:", sum(cosine_scores)/len(cosine_scores))


✅ Saved evaluation metrics CSV to pipeline1_rag_eval_legalbert.csv
Avg ROUGE-L: 0.10007007275374254
Avg Cosine: 0.8648442305051364
